# Extracting LinkedIn Profile Details with Python Automation


### This script automates web scraping tasks using Selenium. It is designed to extract data from specified web pages and store the results in a Pandas DataFrame. Before running the script, ensure that the required packages are installed and the environment is set up properly.


In [79]:
!pip install selenium pandas

In [91]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import json
import time
import pandas as pd

def scrape_linkedin_profile(profile_url, username, password):
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    
    # Go to LinkedIn login page
    driver.get("https://www.linkedin.com/login")
    driver.implicitly_wait(2)

    # Log in to LinkedIn
    try:
        username_field = driver.find_element(By.ID, "username")
        password_field = driver.find_element(By.ID, "password")
        
        username_field.send_keys(username)
        password_field.send_keys(password)

        # Click the login button
        login_button = driver.find_element(By.XPATH, "//button[@type='submit']")
        login_button.click()
    except Exception as e:
        print(f"Error during login: {e}")
        driver.quit()
        return None
    
    time.sleep(1)  # Wait for the page to load after login
    driver.get(profile_url)

    profile_data = {
        "url": profile_url,
        "name": "",
        "headline": "",
        "location": "",
        "contact_number": "",
        "email": "",
        "skills": [],
        "experiences": [],
        "education": [],
        "Summary": "",
        "licenses_and_certifications": [],
        "projects": []
    }

    # Extract name
    try:
        name_element = driver.find_element(By.CSS_SELECTOR, "h1.text-heading-xlarge.inline.t-24.v-align-middle.break-words")
        profile_data["name"] = name_element.text.strip()
    except Exception as e:
        print(f"Error extracting name: {e}")

    # Extract headline
    try:
        description_element = driver.find_element(By.CSS_SELECTOR, ".text-body-medium.break-words")
        profile_data["headline"] = description_element.text.strip()
    except Exception as e:
        print(f"Error extracting headline: {e}")

    # Extract location
    try:
        location_element = driver.find_element(By.CSS_SELECTOR, "span.text-body-small.inline.t-black--light.break-words")
        profile_data['location'] = location_element.text.strip()
    except Exception as e:
        print(f"Error extracting location: {e}")

    # Extract about
    try:
        about_section = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.ID, 'about'))
        )
        target_div = about_section.find_element(By.XPATH, './following-sibling::div/following-sibling::div')
        about_text_element = target_div.find_element(By.CSS_SELECTOR, 'span[aria-hidden="true"]')
        profile_data["Summary"] = about_text_element.text.strip()
    except TimeoutException as e:
        print(f"Error locating About section: {e}")
    except NoSuchElementException as e:
        print(f"Error finding About section: {e}")

    
    # Navigate to contact info overlay
    try:
        contact_info_button = driver.find_element(By.XPATH, "//a[contains(@href, 'overlay/contact-info')]")
        contact_info_button.click()
        
        # Wait for the contact info to load
        time.sleep(2)  # Adjust if necessary
    except Exception as e:
        print(f"Error opening contact info: {e}")

    # Extract contact number
    try:
        contact_number_element = driver.find_element(By.CSS_SELECTOR, "span.t-14.t-black.t-normal")
        number_text = contact_number_element.text.strip()

        # Validate contact number format
        if any(char.isalpha() for char in number_text):
            profile_data['contact_number'] = "Contact number not present"
        else:
            profile_data['contact_number'] = number_text
    except NoSuchElementException:
        profile_data['contact_number'] = "Contact number not present"
    except Exception as e:
        print(f"Error extracting contact number: {e}")

    # Extract email
    try:
        email_element = driver.find_element(By.CSS_SELECTOR, "a[href^='mailto:']")
        profile_data['email'] = email_element.text.strip()
    except NoSuchElementException:
        profile_data['email'] = "Email not present"
    except Exception as e:
        print(f"Error extracting email: {e}")

    # Close the contact info overlay
    driver.back()  # Go back to the profile page

    # Extract experiences
    profile_data["experiences"] = []
    experience_url = f"{profile_url}details/experience/"
    driver.get(experience_url)
    try:
        experience_section = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="profile-content"]/div/div[2]/div/div/main/section/div[2]/div/div[1]/ul'))
        )

        experience_elements = experience_section.find_elements(By.TAG_NAME, "li")
        for element in experience_elements:
            experience = {}
            try:
                title_element = element.find_element(By.CSS_SELECTOR, ".mr1.t-bold span")
                experience["title"] = title_element.text.strip()
            except Exception as e:
                print(f"Error extracting title for experience: {e}")
                continue

            try:
                company_element = element.find_element(By.CSS_SELECTOR, "a.optional-action-target-wrapper")
                company_name = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal span.visually-hidden").text.strip()
                company_link = company_element.get_attribute("href")
                experience["company_link"] = company_link
                experience["company_name"] = company_name    
            except Exception as e:
                print(f"Error extracting company information: {e}")
                continue

            try:
                duration_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal.t-black--light span.pvs-entity__caption-wrapper")
                experience["duration"] = duration_element.text.strip()
            except Exception as e:
                print(f"Error extracting duration: {e}")

            try:
                location_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal.t-black--light span.visually-hidden")
                experience["location"] = location_element.text.strip()
            except Exception as e:
                print(f"Error extracting location: {e}")

            try:
                skills_element = element.find_element(By.CSS_SELECTOR, ".pvs-entity__sub-components span[aria-hidden='true']")
                experience["skills/description"] = skills_element.text.strip()
            except Exception as e:
                print(f"Error extracting skills: {e}")

            profile_data["experiences"].append(experience)
    except TimeoutException as e:
        print(f"Error finding experiences: {e}")
    except NoSuchElementException as e:
        print(f"Error finding experiences: {e}")


    # Extract education
    profile_data["education"] = []
    education_url = f"{profile_url}details/education/"
    driver.get(education_url)
    try:
        education_section = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="profile-content"]/div/div[2]/div/div/main/section/div[2]/div/div[1]/ul'))
        )

        education_elements = education_section.find_elements(By.TAG_NAME, "li")
        for element in education_elements:
            education = {}
            try:
                school_element = element.find_element(By.CSS_SELECTOR, ".mr1.hoverable-link-text.t-bold span")
                education["school"] = school_element.text.strip()
            except Exception as e:
                print(f"Error extracting school for education: {e}")
                continue

            try:
                degree_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal span[aria-hidden='true']")
                education["degree"] = degree_element.text.strip()
            except Exception as e:
                print(f"Error extracting degree for education: {e}")

            try:
                dates_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal.t-black--light span.pvs-entity__caption-wrapper")
                education["dates"] = dates_element.text.strip()
            except Exception as e:
                print(f"Error extracting dates for education: {e}")

            try:
                grade_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal.t-black span[aria-hidden='true']")
                education["grade"] = grade_element.text.strip()
            except Exception as e:
                print(f"Error extracting grade for education: {e}")

            profile_data["education"].append(education)
    except TimeoutException as e:
        print(f"Error finding education: {e}")
    except NoSuchElementException as e:
        print(f"Error finding education: {e}")

    # Extract licenses and certifications
    driver.get(f"{profile_url}details/certifications/")
    try:
        licenses_certifications_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="profile-content"]/div/div[2]/div/div/main/section/div[2]/div/div[1]/ul'))
        )
        licenses_certifications_elements = licenses_certifications_section.find_elements(By.TAG_NAME, "li")
        for element in licenses_certifications_elements:
            license_certification = {}
            try:
                name_element = element.find_element(By.CSS_SELECTOR, ".mr1.t-bold span")
                license_certification["name"] = name_element.text.strip()
            except Exception as e:
                print(f"Error extracting name for license/certification: {e}")
                continue

            try:
                organization_element = element.find_element(By.CSS_SELECTOR, "a.optional-action-target-wrapper")
                organization_link = organization_element.get_attribute("href")
                organization_name = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal span.visually-hidden").text.strip()
                license_certification["organization_link"] = organization_link
                license_certification["organization_name"] = organization_name
            except Exception as e:
                print(f"Error extracting organization info for license/certification: {e}")

            try:
                dates_element = element.find_element(By.CSS_SELECTOR, ".t-14.t-normal.t-black--light span.pvs-entity__caption-wrapper")
                license_certification["dates"] = dates_element.text.strip()
            except Exception as e:
                print(f"Error extracting dates for license/certification: {e}")

            profile_data["licenses_and_certifications"].append(license_certification)
    except TimeoutException as e:
        print(f"Error finding licenses and certifications: {e}")




       # Extract projects
    driver.get(f"{profile_url}details/projects/")
    try:
        projects_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="profile-content"]/div/div[2]/div/div/main/section/div[2]/div/div[1]/ul'))
        )
        projects_elements = projects_section.find_elements(By.TAG_NAME, "li")
        for element in projects_elements:
            project = {}
            try:
                project_name_element = element.find_element(By.CSS_SELECTOR, ".mr1.t-bold span")
                project["name"] = project_name_element.text.strip()
            except Exception as e:
                print(f"Error extracting name for project: {e}")

            profile_data["projects"].append(project)
    except TimeoutException as e:
        print(f"Error finding projects: {e}")

 
    # Extract skills using the enhanced code
    skills_url = f"{profile_url}details/skills/"
    driver.get(skills_url)

    skills = []  # Initialize skills list
    try:
        # Wait for the skills section to load
        skills_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//section[contains(@class, 'artdeco-card')]"))
        )

        # Scroll down to load all skills
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            # Scroll down to the bottom
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)  # Wait for new skills to load
            
            # Calculate new scroll height and compare with last scroll height
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:  # If heights are the same, we've reached the bottom
                break
            last_height = new_height

        # Extract skills dynamically
        max_retries = 3  # Retry logic to handle stale element errors
        retries = 0
        while retries < max_retries:
            try:
                # Find all skill elements
                skill_elements = skills_section.find_elements(By.XPATH, ".//span[contains(@class, 'visually-hidden')]")
                for skill_element in skill_elements:
                    skill_name = skill_element.text.strip()
                    if skill_name not in skills:  # Avoid duplicates
                        skills.append(skill_name)
                break  # Exit retry loop if successful
            except Exception as e:
                print(f"Error extracting skills: {e}")
                retries += 1

        profile_data["skills"] = skills
    except TimeoutException as e:
        print(f"Error loading skills section: {e}")

    # Convert profile data to Pandas DataFrame
    profile_df = pd.DataFrame([profile_data])

    # Save the DataFrame to a CSV file
    profile_df.to_csv("linkedin_profile_data.csv", index=False)

    driver.quit()
    return profile_df  # Return the DataFrame instead of the dictionary

# Example usage
profile_url = "https://www.linkedin.com/in/dr-saurabh-shukla-6767b5116/"  # Replace with the desired profile URL
username = "nikhilkumar2448@gmail.com"  # Replace with your LinkedIn username
password = "nikhilkumar2448"  # Replace with your LinkedIn password

profile_df = scrape_linkedin_profile(profile_url, username, password)
print(profile_df)


Error extracting title for experience: Message: no such element: Unable to locate element: {"method":"css selector","selector":".mr1.t-bold span"}
  (Session info: chrome=130.0.6723.59); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF690373AB5+28005]
	(No symbol) [0x00007FF6902D83B0]
	(No symbol) [0x00007FF69017580A]
	(No symbol) [0x00007FF6901C5A3E]
	(No symbol) [0x00007FF6901C5D2C]
	(No symbol) [0x00007FF6901B937C]
	(No symbol) [0x00007FF6901EBA7F]
	(No symbol) [0x00007FF6901B9246]
	(No symbol) [0x00007FF6901EBC50]
	(No symbol) [0x00007FF69020B8B3]
	(No symbol) [0x00007FF6901EB7E3]
	(No symbol) [0x00007FF6901B75C8]
	(No symbol) [0x00007FF6901B8731]
	GetHandleVerifier [0x00007FF69066643D+3118829]
	GetHandleVerifier [0x00007FF6906B6C90+3448640]
	GetHandleVerifier [0x00007FF6906ACF0D+3408317]
	GetHandleVerifier [0x00007FF69043A40B+841403]
	(No symbol

In [93]:
profile_df

,url,name,headline,location,contact_number,email,skills,experiences,education,Summary,licenses_and_certifications,projects
0,https://www.linkedin.com/in/dr-saurabh-shukla-...,Dr. Saurabh Shukla,Assistant Professor @ Indian Institute of Info...,"Lucknow, Uttar Pradesh, India",Contact number not present,saurabhshkl.shukla@gmail.com,"[Blockchain, 2 experiences across Indian Insti...","[{'title': 'Head of Department', 'company_link...","[{'school': 'Universiti Teknologi PETRONAS', '...","Assistant Professor and Head of Department, Co...",[{'name': 'Verified International Academic Qua...,[{'name': 'Cooperative Energy Trading System (...


In [95]:
profile_df.name[0]

'Dr. Saurabh Shukla'

In [97]:
profile_df.headline[0]

'Assistant Professor @ Indian Institute of Information Technology Lucknow (IIITL), PhD (UTP, Malaysia), Post Doc (NUIG, Ireland)'

In [99]:
profile_df.contact_number[0]

'Contact number not present'

In [101]:
profile_df.email[0]

'saurabhshkl.shukla@gmail.com'

In [103]:
profile_df.skills[0]

['Blockchain',
 '2 experiences across Indian Institute of Information Technology Lucknow and 1 other company',
 '1 endorsement',
 'Ethereum',
 'Postdoctoral Research Scientist at University of Galway',
 'Smart Contracts',
 'Smart Grid',
 'Cybersecurity',
 'Python (Programming Language)',
 'Head of Department at Indian Institute of Information Technology Lucknow',
 '2 endorsements',
 'C',
 'C++',
 'Programming',
 'Research',
 'Project Management',
 'Algorithms',
 'Data Structures',
 'Machine Learning',
 'Artificial Intelligence (AI)',
 'Statistics',
 'Microsoft Excel',
 '3 endorsements',
 'Microsoft Office',
 'PowerPoint',
 'Java',
 'Microsoft Word',
 'MATLAB',
 'Core Java',
 'Recurrent Neural Networks (RNN)',
 'Deep Neural Networks (DNN)',
 'Keras',
 'PyTorch',
 'TensorFlow',
 'Management',
 'Customer Service',
 'Leadership',
 'Public Speaking',
 'Deep Learning',
 'Data Science',
 'Data Analytics',
 'Applied Machine Learning',
 'Neural Networks',
 'English',
 'Programming Languages',
 

In [105]:
profile_df.experiences[0]

[{'title': 'Head of Department',
  'company_link': 'https://www.linkedin.com/company/13705261/',
  'company_name': 'Indian Institute of Information Technology Lucknow · Full-time',
  'duration': 'Aug 2023 - Present · 1 yr 3 mos',
  'location': 'Aug 2023 to Present · 1 yr 3 mos',
  'skills/description': 'Skills: C++ · Python (Programming Language) · Machine Learning · Blockchain'},
 {'title': 'Assistant Professor',
  'company_link': 'https://www.linkedin.com/company/18657381/',
  'company_name': 'Amity University Mumbai · Full-time',
  'duration': 'Sep 2022 - Jul 2023 · 11 mos',
  'location': 'Sep 2022 to Jul 2023 · 11 mos'},
 {'title': 'Postdoctoral Research Scientist',
  'company_link': 'https://www.linkedin.com/company/7899/',
  'company_name': 'University of Galway · Full-time',
  'duration': 'Sep 2020 - Aug 2022 · 2 yrs',
  'location': 'Sep 2020 to Aug 2022 · 2 yrs',
  'skills/description': 'Data Science Institute, NUIG'},
 {'title': 'Teaching Fellow',
  'company_link': 'https://ww

In [107]:
profile_df.education[0]

[{'school': 'Universiti Teknologi PETRONAS',
  'degree': 'Doctor of Philosophy - PhD, Information Technology',
  'dates': '2017 - 2020',
  'grade': 'Grade: GRADUATE ON TIME (GOT) - A'},
 {'school': 'Indian Institute Of Information Technology Allahabad',
  'degree': 'Master of Technology - MTech, Information Technology',
  'dates': 'Jul 2008 - Jun 2010',
  'grade': 'Grade: A'},
 {'school': 'Dr. A.P.J. Abdul Kalam Technical University',
  'degree': 'Bachelor of Technology - BTech, Information Technology',
  'dates': '2004 - 2008',
  'grade': 'Grade: A'}]

In [109]:
profile_df.Summary[0]

'Assistant Professor and Head of Department, Computer Science at the Indian Institute of Information Technology Lucknow (IIITL), Postdoctoral Researcher, Unit of Semantic Web, Data Science Institute (DSI), Insight SFI Centre of Data Analytics, National University of Ireland Galway (NUIG), Galway, Ireland, PhD, University Teknologi PETRONAS (UTP), Malaysia, Alumni IIITA.'

In [111]:
profile_df.licenses_and_certifications[0]

[{'name': 'Verified International Academic Qualifications',
  'organization_link': 'https://www.linkedin.com/company/61690/',
  'organization_name': 'World Education Services',
  'dates': 'Issued Jul 2020'}]

In [121]:
profile_df.projects[0]

[{'name': 'Cooperative Energy Trading System (CENTS)'},
 {},
 {},
 {},
 {'name': 'An Efficient and Scalable Architecture for Healthcare Internet-of-Things'},
 {},
 {},
 {},
 {},
 {}]